# LegalBERT Finetuning and testing
made with help from:
- https://github.com/cltl/ma-ml4nlp-labs/blob/main/code/assignment3/other_systems/bert_finetunen.ipynb

In [ ]:
import torch
from transformers import AutoTokenizer, Trainer, TrainingArguments,
from datasets import load_dataset, DatasetDict
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import AutoTokenizer
from transformers import Trainer
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
import os
from datasets import Dataset

In [ ]:
# ! pip install scikit-learn
import torch
import transformers
from transformers import pipeline
from datasets import load_dataset
from evaluate import load
import pandas as pd
import sklearn
from datasets import Dataset
from transformers import RobertaTokenizerFast
import numpy as np
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, AutoTokenizer
from transformers import DataCollatorForTokenClassification
from seqeval.metrics import classification_report 

In [2]:
task = "ner"
model_checkpoint ="nlpaueb/legal-bert-base-uncased"
batch_size = 16

In [4]:
# file = (r".\combined_opensource.conll")
# train_file = r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya\IAE_train.conll"
# val_file =r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya\IAE_val.conll"
train_file = r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\fixed_combined_training_set.conll"
val_file = r"C:\Users\M.Walavalkar\OneDrive - IBFD\Desktop\thesis-ner-manya-explore\fixed_combined_validation_set.conll"


In [ ]:
def get_list_of_sentences(file_path):
    """this returns a list of lists for sentences and list of lists for labels"""
    current_sent = []
    current_label = []
    all_sents = []
    all_labels = []

    with open(file_path, "r", encoding="utf-8") as infile:
        lines = infile.readlines()
        for line in lines:
            line = line.strip()
            if line == "":
                if len(current_sent) > 0:
                    all_sents.append(current_sent)
                    all_labels.append(current_label)
                    current_sent = []
                    current_label = []
            else:
                splitted = line.split("\t")
                token_part = splitted[0]
                label_part = splitted[1]
                current_sent.append(token_part)
                current_label.append(label_part)

        if len(current_sent) > 0:
            all_sents.append(current_sent)
            all_labels.append(current_label)
    
    return all_sents, all_labels

In [ ]:
train_tokens, train_labels = get_list_of_sentences(train_file)
val_tokens, val_labels = get_list_of_sentences(val_file)

train_dataset = Dataset.from_dict({"tokens": train_tokens, "ner_tags": train_labels})
val_dataset = Dataset.from_dict({"tokens": val_tokens, "ner_tags": val_labels})

In [12]:
label_list = sorted(set(label for sentence in train_labels for label in sentence))
label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for label, i in label_to_id.items()}

In [13]:
tokenizer = AutoTokenizer.from_pretrained('nlpaueb/legal-bert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

c:\Users\M.Walavalkar\AppData\Local\anaconda3\envs\thesis_ner\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\M.Walavalkar\.cache\huggingface\hub\models--nlpaueb--legal-bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
def tokenize_and_align_labels(examples):
    """
    this tokenizes the sentences and aligns the labels with the tokenized subwords
    """
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    #tokenized the lists of sentences
    all_labels = []
    for i, word_labels in enumerate(examples["ner_tags"]): #for every label list in the ner_tags we have
        word_ids = tokenized_inputs.word_ids(batch_index=i) #mapping every subword to the original word index
        aligned_labels = [] 
        previous_word_id = None #looking at the previous word id to detect new ones
        for word_id in word_ids: #going through each token
            #special tokens have a None word id, they are set to -100 to be ignored
            #in the loss function
            if word_id is None: #special tokens
                aligned_labels.append(-100)
            elif word_id != previous_word_id: #if not equal to same as prev, new word!
                aligned_labels.append(label_to_id[word_labels[word_id]]) #assign it the gold lab of the first subword
            else: #continued subword of the same word
                label = word_labels[word_id] #getting the OG label
                if label.startswith("B-"): #check if the label begins a NE
                    aligned_labels.append(label_to_id["I-" + label[2:]]) #changing the rest to I-
                else:
                    aligned_labels.append(label_to_id[label]) #if label already an I- or O it stays the same
            previous_word_id = word_id #update the prev word for next iteration

        all_labels.append(aligned_labels) #appending list of labels to overall list

    tokenized_inputs["labels"] = all_labels #add the aligned labels to the tokenized input
    return tokenized_inputs

In [19]:
tokenized_train = train_dataset.map(tokenize_and_align_labels, batched=True).remove_columns(["tokens", "ner_tags"])
tokenized_val = val_dataset.map(tokenize_and_align_labels, batched=True).remove_columns(["tokens", "ner_tags"])

Map:   0%|          | 0/30562 [00:00<?, ? examples/s]

Map:   0%|          | 0/3396 [00:00<?, ? examples/s]

In [ ]:
SEED = 67
set_seed(SEED)
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(label_list), id2label=id_to_label, label2id=label_to_id)
model_name = model_checkpoint.split("/")[-1]

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./legalbert-ner",
    eval_strategy="epoch",       
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,  
    metric_for_best_model="eval_loss",
    seed=SEED,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,  
    processing_class=tokenizer,
    data_collator=data_collator
)

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [24]:
trainer.train()

c:\Users\M.Walavalkar\AppData\Local\anaconda3\envs\thesis_ner\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
trainer.save_model("./finetuned_roberta_fixed_opensource")
tokenizer.save_pretrained("./finetuned_roberta_fixed_opensource")

('./finetuned_roberta_IAE\\tokenizer_config.json',
 './finetuned_roberta_IAE\\special_tokens_map.json',
 './finetuned_roberta_IAE\\vocab.json',
 './finetuned_roberta_IAE\\merges.txt',
 './finetuned_roberta_IAE\\added_tokens.json',
 './finetuned_roberta_IAE\\tokenizer.json')

## Testing

In [ ]:
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LEGALBERT_OS/model_output/checkpoint-5328")
tokenizer = AutoTokenizer.from_pretrained(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LEGALBERT_OS/model_output/checkpoint-5328", add_prefix_space=True)
id_to_label = model.config.id2label

In [ ]:
def predict_sentence(tokens):
    """
    this returns a label for each input token in a sentence
    - claude troubleshooting used
    """
    #tokenizing the text
    encoding = tokenizer(tokens, is_split_into_words=True, return_tensors="pt", truncation=True, max_length=512)
    word_ids = encoding.word_ids() #mapping each subword to its original word
    encoding_on_device = {key: val.to(model.device) for key, val in encoding.items()} #needed because tensor error otherwise

    with torch.no_grad():
        outputs = model(**encoding_on_device) #dont need to calculate gradients

    predicted_ids = torch.argmax(outputs.logits, dim=-1)[0].tolist() #argmax to get the label with the highest score
    predicted_subword_labels = [id_to_label[i] for i in predicted_ids] #convert numeric labels to text labels

    word_level_labels = [] #storing one label per original word
    previous_word_id = None

    for subword_index, word_id in enumerate(word_ids):
        if word_id is None: #skipping special tokens
            continue
        if word_id != previous_word_id: #checking if its a first subword of a word
            word_level_labels.append(predicted_subword_labels[subword_index]) #then add the pred for the first subword to the list
        previous_word_id = word_id #update so the next subword of the same word can be skipped

    if len(word_level_labels) < len(tokens): #adding padding to the list 
        word_level_labels += ["O"] * (len(tokens) - len(word_level_labels))

    return word_level_labels

def predict_and_save_conll(test_sents, test_labels, output_path):
    """
    this runs predictions on all sentences and then saves results to a conll file
    """
    true_labels = []
    predicted_labels = []

    with open(output_path, "w", encoding="utf-8") as infile:
        for tokens, gold_labels in zip(test_sents, test_labels):
            predicted = predict_sentence(tokens)
            true_labels.append(gold_labels)
            predicted_labels.append(predicted)

            for token, gold, pred in zip(tokens, gold_labels, predicted):
                infile.write(f"{token}\t{pred}\n")
            infile.write("\n")

    print(f"Predictions saved to: {output_path}")
    return true_labels, pred_labels

In [ ]:
def get_list_of_sentences(file_path):
    """this returns a list of lists for sentences and list of lists for labels"""
    current_sent = []
    current_label = []
    all_sents = []
    all_labels = []

    with open(file_path, "r", encoding="utf-8") as infile:
        lines = infile.readlines()
        for line in lines:
            line = line.strip()
            if line == "":
                if len(current_sent) > 0:
                    all_sents.append(current_sent)
                    all_labels.append(current_label)
                    current_sent = []
                    current_label = []
            else:
                splitted = line.split("\t")
                token_part = splitted[0]
                label_part = splitted[1]
                current_sent.append(token_part)
                current_label.append(label_part)

        if len(current_sent) > 0:
            all_sents.append(current_sent)
            all_labels.append(current_label)
    
    return all_sents, all_labels

In [ ]:
test_sents, test_labels = get_list_of_sentences(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/datasets/my_data/final_dataset_1st_may.conll")

In [8]:
model = AutoModelForTokenClassification.from_pretrained(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LEGALBERT_OS/model_output/checkpoint-5328")

id_to_label = model.config.id2label
output_conll = r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LEGALBERT_OS/model_output/checkpoint-5328-preds"
true_labels, pred_labels = predict_and_save_conll(test_sents, test_labels, output_conll)

Predictions saved to: /Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LEGALBERT_OS/model_output/checkpoint-5328-preds


In [9]:
##legalbert -- existing legal data
pred_sents, pred_labels = get_list_of_sentences(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LEGALBERT_OS/model_output/checkpoint-5328-preds")
true_labels = []
pred_labels = []

for tokens, gold_labels in zip(test_sents, test_labels):
    predicted = predict_sentence(tokens)
    true_labels.append(gold_labels)
    pred_labels.append(predicted)

report = classification_report(true_labels, pred_labels)
print(report)

              precision    recall  f1-score   support

       COURT       0.57      0.41      0.47       106
        DATE       0.54      0.20      0.29       132
         GPE       0.66      0.54      0.59       299
JURISDICTION       0.00      0.00      0.00       153
         LAW       0.07      0.15      0.09        65
         ORG       0.54      0.17      0.26       221
      PERSON       0.78      0.64      0.70       186
   PROVISION       0.14      0.16      0.15        87
 TAX_CONCEPT       0.00      0.00      0.00       353
    TAX_TYPE       0.00      0.00      0.00        86

   micro avg       0.49      0.24      0.32      1688
   macro avg       0.33      0.23      0.26      1688
weighted avg       0.36      0.24      0.28      1688



/opt/anaconda3/lib/python3.13/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [10]:
model = AutoModelForTokenClassification.from_pretrained(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LEGALBERT_LLM/model_output/checkpoint-4008")

id_to_label = model.config.id2label
output_conll = r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LEGALBERT_LLM/model_output/checkpoint-4008-preds"
true_labels, pred_labels = predict_and_save_conll(test_sents, test_labels, output_conll)

Predictions saved to: /Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LEGALBERT_LLM/model_output/checkpoint-4008-preds


In [11]:
##legalbert -- LLM labeled data
pred_sents, pred_labels = get_list_of_sentences(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LEGALBERT_LLM/model_output/checkpoint-4008-preds")
true_labels = []
pred_labels = []

for tokens, gold_labels in zip(test_sents, test_labels):
    predicted = predict_sentence(tokens)  
    true_labels.append(gold_labels)
    pred_labels.append(predicted)

report = classification_report(true_labels, pred_labels)
print(report)

              precision    recall  f1-score   support

       COURT       0.41      0.79      0.54       106
        DATE       0.76      0.80      0.78       132
         GPE       0.58      0.84      0.68       299
JURISDICTION       0.57      0.82      0.67       153
         LAW       0.19      0.51      0.28        65
         ORG       0.42      0.57      0.48       221
      PERSON       0.43      0.54      0.48       186
   PROVISION       0.36      0.67      0.47        87
 TAX_CONCEPT       0.31      0.21      0.25       353
    TAX_TYPE       0.36      0.55      0.44        86

   micro avg       0.45      0.59      0.51      1688
   macro avg       0.44      0.63      0.51      1688
weighted avg       0.45      0.59      0.50      1688



In [12]:
model = AutoModelForTokenClassification.from_pretrained(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LEGALBERT_MIXED/model_output/checkpoint-9711")

id_to_label = model.config.id2label
output_conll = r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LEGALBERT_MIXED/model_output/checkpoint-9711-preds"
true_labels, pred_labels = predict_and_save_conll(test_sents, test_labels, output_conll)

Predictions saved to: /Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LEGALBERT_MIXED/model_output/checkpoint-9711-preds


In [ ]:
##legalbert --  mixed data
pred_sents, pred_labels = get_list_of_sentences(r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/LEGALBERT_MIXED/model_output/checkpoint-9711-preds")
true_labels = []
pred_labels = []

for tokens, gold_labels in zip(test_sents, test_labels):
    predicted = predict_sentence(tokens)  
    true_labels.append(gold_labels)
    pred_labels.append(predicted)

report = classification_report(true_labels, pred_labels)
print(report)

              precision    recall  f1-score   support

       COURT       0.40      0.77      0.53       106
        DATE       0.75      0.75      0.75       132
         GPE       0.66      0.85      0.74       299
JURISDICTION       0.57      0.81      0.67       153
         LAW       0.14      0.45      0.21        65
         ORG       0.45      0.50      0.47       221
      PERSON       0.33      0.44      0.38       186
   PROVISION       0.39      0.60      0.47        87
 TAX_CONCEPT       0.28      0.18      0.22       353
    TAX_TYPE       0.45      0.52      0.48        86

   micro avg       0.45      0.56      0.50      1688
   macro avg       0.44      0.59      0.49      1688
weighted avg       0.45      0.56      0.49      1688

